# 73 — Blind-A submission: multi-modal Stage A + Stage B cross-encoder (config 182) → prediction.zip

Produces `prediction.json` for CodaBench by running the **Phase A + Phase B** pipeline
over the 80 Blind-A queries:

```
wRRF(BM25 + multi-modal dense bi-encoder) → multi-modal cross-encoder reranker → v5-kto Qwen-3B responder
```

**Config 182** points at the multi-modal Stage A retriever
(`OrRim123/recsys2026-bge-base-en-music-v1-mm-merged`, nb 70) and the Stage B
multi-modal cross-encoder reranker (`OrRim123/recsys2026-mm-reranker-v1`, nb 71),
with the v5-kto Qwen-3B responder. CMQR is OFF (training queries are structured
multi-turn blobs; CMQR paraphrases would break the train/inference match).

Run order: 1 CONFIG → 2 setup → 3 preflight+inference → 4 validate → 5 zip.
Expected wallclock: ~35-65 min on Blackwell.

> Prereqs (preflight enforces them): nb 70 pushed the merged multi-modal Stage A
> AND embedded the catalog with it; nb 71 finished + pushed the reranker; the
> Phase 0 multi-modal artifacts dir is on Drive.

In [ ]:
# 1) CONFIG — single source of truth for this submission.
# Submission 3: multi-modal Stage A retriever (nb 70) + multi-modal Stage B
# cross-encoder reranker (nb 71) + v5-kto Qwen-3B responder.

# --- Identity --------------------------------------------------
BRANCH         = 'stage-b-cross-encoder'
TID            = '182-mm-stagea-ce-v5kto-blindA'     # config: music-crs-baselines/config/<TID>.yaml
HUB_USER       = 'OrRim123'

# --- Stage A (multi-modal bi-encoder) — MUST match nb 70/71 ----
STAGE_A_REPO   = f'{HUB_USER}/recsys2026-bge-base-en-music-v1-mm-merged'
EMBED_LABEL    = 'bge-base-en-music-v1-mm-merged'    # catalog-embed label (nb 71 EMBED_LABEL)

# --- Stage B (multi-modal cross-encoder reranker) — from nb 71 -
RERANKER_REPO  = f'{HUB_USER}/recsys2026-mm-reranker-v1'

# --- Shared multi-modal artifacts (same dir nb 70/71 used) -----
MULTIMODAL_ARTIFACTS = '/content/drive/MyDrive/recsys2026_retrieval_v2_cache/multimodal'

# --- Derived paths (do not edit usually) -----------------------
# Catalog pickle path mirrors DENSE_MULTIMODAL_LOCAL:
#   {cache_dir}/dense_local/{safe_model}/{embed_label}/track_embeddings.pkl
# nb 73 cell 2 symlinks experiments/cache/dense_local -> the Drive dir below.
SAFE_MODEL        = STAGE_A_REPO.replace('/', '_')
CATALOG_PKL       = f'/content/drive/MyDrive/recsys2026_retrieval_v2_cache/dense_local/{SAFE_MODEL}/{EMBED_LABEL}/track_embeddings.pkl'
PRED_PATH         = f'music-crs-baselines/exp/inference/blindset_A/{TID}.json'
INFERENCE_LOG     = f'/content/drive/MyDrive/recsys2026_retrieval_v2_cache/blindA_{TID}_log.txt'
SUBMISSION_DATE   = None   # auto-fills with today; override with 'YYYY-MM-DD' for replays
SUBMISSION_LABEL  = f'retrieval-v2-{TID}'

# --- Inference knobs ------------------------------------------
INFERENCE_BATCH_SIZE = 32

print('Config loaded:')
for k in ('BRANCH','TID','STAGE_A_REPO','RERANKER_REPO','MULTIMODAL_ARTIFACTS','CATALOG_PKL','PRED_PATH','INFERENCE_BATCH_SIZE'):
    print(f'  {k} = {globals()[k]!r}')

In [ ]:
# 2) Setup — clone branch + HF auth + Drive mount + symlinks + deps.
import os
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive', force_remount=False)

!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
for name, drive_subdir in [
    ('sid', 'recsys2026_sid_cache'),
    ('dense', 'recsys2026_dense_cache'),
    # Fine-tuned BGE-M3 catalog pickle: written by nb 70 cell 6 to
    # MyDrive/recsys2026_retrieval_v2_cache/dense_local/. Without this
    # symlink, DENSE_LOCAL FileNotFoundError on inference.
    ('dense_local', 'recsys2026_retrieval_v2_cache/dense_local'),
]:
    src = f'{DRIVE_BASE}/{drive_subdir}'
    dst = f'{LOCAL_BASE}/{name}'
    os.makedirs(src, exist_ok=True)
    if os.path.islink(dst): os.unlink(dst)
    elif os.path.exists(dst):
        import shutil; shutil.rmtree(dst)
    os.symlink(src, dst)

# Inference deps. Mirrors notebook 41 cell 5's install set.
!pip install -q --upgrade \
    'peft>=0.10' 'transformers>=4.40' 'accelerate>=0.30' 'torchao>=0.17' \
    'bm25s>=0.3.0,<0.4' 'sentence-transformers' \
    'datasets' 'pandas<3.0' 'tqdm' 'omegaconf' 'pyyaml' \
    'trl>=0.12.0'


In [ ]:
# 3) Preflight + inference. Validates every required artifact before spending
# ~45 min on Blind-A inference; clear error if anything's missing.
import os
from huggingface_hub import HfApi
from pathlib import Path

api = HfApi()

# Preflight 1: Stage A multi-modal merged model on Hub (nb 70 pushes it).
try:
    api.model_info(STAGE_A_REPO)
    print(f'[preflight] OK Stage A model reachable: {STAGE_A_REPO}')
except Exception as e:
    raise RuntimeError(
        f'[preflight] Stage A model NOT reachable: {STAGE_A_REPO!r}\n'
        f'  Cause: nb 70 has not pushed the merged multi-modal model yet.\n  {e}')

# Preflight 2: Stage B multi-modal cross-encoder reranker on Hub (nb 71 cell 4).
try:
    api.model_info(RERANKER_REPO)
    print(f'[preflight] OK Stage B reranker reachable: {RERANKER_REPO}')
except Exception as e:
    raise RuntimeError(
        f'[preflight] Stage B reranker NOT reachable: {RERANKER_REPO!r}\n'
        f'  Cause: nb 71 cell 4 has not finished + pushed the reranker yet.\n  {e}')

# Preflight 3: multi-modal catalog pickle on Drive (embed_catalog_multimodal.py).
if not Path(CATALOG_PKL).exists():
    raise FileNotFoundError(
        f'[preflight] Multi-modal catalog pickle MISSING: {CATALOG_PKL}\n'
        f'  Encode the catalog with the Stage A multi-modal model first:\n'
        f'    python scripts/embed_catalog_multimodal.py --model-dir {STAGE_A_REPO} \\\n'
        f'      --multimodal-artifacts {MULTIMODAL_ARTIFACTS} \\\n'
        f'      --catalog-out-dir <cache>/dense_local/{SAFE_MODEL}/{EMBED_LABEL}')
_sz = Path(CATALOG_PKL).stat().st_size / (1024 * 1024)
print(f'[preflight] OK multi-modal catalog pickle exists ({_sz:.1f} MB)')

# Preflight 4: multi-modal artifacts dir — needed by BOTH retriever and reranker.
_art = Path(MULTIMODAL_ARTIFACTS)
_need = ['tag_vocab.json', 'user_cf.npy']
_missing = [f for f in _need if not (_art / f).exists()]
if not _art.is_dir() or _missing:
    raise FileNotFoundError(
        f'[preflight] Multi-modal artifacts incomplete at {MULTIMODAL_ARTIFACTS} '
        f'(missing: {_missing or "the dir itself"}). Re-run the Phase 0 precompute (nb 70).')
print(f'[preflight] OK multi-modal artifacts present: {MULTIMODAL_ARTIFACTS}')

# Preflight 5: config file exists.
_config_path = Path(f'music-crs-baselines/config/{TID}.yaml')
if not _config_path.exists():
    raise FileNotFoundError(
        f'[preflight] Config MISSING: {_config_path}. Check TID in the CONFIG cell.')
print(f'[preflight] OK config file exists: {_config_path}')

print(f'\n[preflight] all checks passed — kicking off Blind-A inference (~30-60 min on Blackwell)')
print(f'[preflight] live log streaming to {INFERENCE_LOG}')
print(f'[preflight] tail -f that file in a separate Colab tab to watch progress.\n')

%cd /content/recsys2026/music-crs-baselines
!python run_inference_blindset.py \
    --tid {TID} \
    --batch_size {INFERENCE_BATCH_SIZE} \
    2>&1 | tee {INFERENCE_LOG} | tail -80

In [ ]:
# 4) Validate prediction.json (schema + record count + catalog-membership).
%cd /content/recsys2026
import json, subprocess, sys
from pathlib import Path
sys.path.insert(0, '/content/recsys2026')
from scripts.precheck_prediction import precheck, _load_catalog

preds = json.load(open(PRED_PATH))
n_entries = len(preds) if isinstance(preds, list) else len(preds.keys())
print(f'[validate] prediction file: {n_entries} entries')
assert n_entries == 80, f'EXPECTED 80, got {n_entries} — DO NOT submit'
sample_keys = list(preds[0].keys()) if isinstance(preds, list) else list(list(preds.values())[0].keys())
print(f'[validate] sample entry keys: {sample_keys}')

# Strict precheck: catalog membership + schema invariants.
catalog = _load_catalog('talkpl-ai/TalkPlayData-Challenge-Track-Metadata')
result = precheck(Path(PRED_PATH), catalog=catalog, expected_n=80)
assert result['ok'], (
    f'precheck FAILED with {len(result["errors"])} errors; '
    f'first 5: {result["errors"][:5]}'
)
print(f'[validate] ✓ precheck passed (n_records={result["n_records"]})')

# Schema validator (catches different bugs from precheck).
rc = subprocess.call(['python', 'scripts/validate_prediction.py', '--input', PRED_PATH, '--split', 'blindA'])
assert rc == 0, f'validate_prediction.py FAILED (rc={rc}) — do not submit'
print('[validate] ✓ schema validator passed')

# Peek at the first prediction to eyeball quality.
sample = preds[0] if isinstance(preds, list) else list(preds.values())[0]
print(f'\n[validate] first prediction sample:')
for k, v in sample.items():
    if isinstance(v, str) and len(v) > 200:
        print(f'  {k}: {v[:200]}...')
    elif isinstance(v, list) and len(v) > 5:
        print(f'  {k}: {v[:5]} ... ({len(v)} items)')
    else:
        print(f'  {k}: {v!r}')


In [ ]:
# 5) Zip for CodaBench. prediction.json MUST be at the ROOT of the zip
# (per memory project_codabench_submission: server reads /app/input/res/prediction.json).
import os, zipfile, datetime
_date = SUBMISSION_DATE or datetime.date.today().strftime('%Y-%m-%d')
zip_path = f'/content/drive/MyDrive/recsys2026_submissions/{_date}-{SUBMISSION_LABEL}.zip'
os.makedirs(os.path.dirname(zip_path), exist_ok=True)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_PATH, arcname='prediction.json')
print(f'[zip] ready: {zip_path}')
print(f'[zip] size: {os.path.getsize(zip_path) / 1024:.1f} KB')
print(f'\nDownload + upload to CodaBench at https://www.codabench.org/competitions/\n'
      f'After scoring, log the result via:\n'
      f'  !python scripts/blind_a_score_tracker.py append --tid {TID} \\\n'
      f'      --composite <X> --ndcg <Y> --llm <Z> --lex_div <W> --cat_div <V>')


## After the run

1. Download the zip from Drive: `/content/drive/MyDrive/recsys2026_submissions/<date>-retrieval-v2-{TID}.zip`
2. Upload to CodaBench (https://www.codabench.org/competitions/).
3. Append the scores to the tracker:
   ```
   !python scripts/blind_a_score_tracker.py append --tid <TID> \
       --composite <X> --ndcg <Y> --llm <Z> --lex_div <W> --cat_div <V>
   ```
4. Compare nDCG@20 against the config-180 (text-only Stage A + ProRank) and
   config-181 submissions to isolate the multi-modal Stage A + cross-encoder lift.
   In-training signals were: Stage A dev nDCG@20 ~0.16 (nb 70); Stage B cascade
   gate +0.04 over Stage A on dev (nb 71 cell 6).
5. If nDCG@20 regressed: most likely a stale catalog embedding (re-embed with the
   CURRENT Stage A model via scripts/embed_catalog_multimodal.py) or a Stage A
   recall@100 ceiling the reranker cannot lift — check recall@100 first.